# Coverages

This notebook serves for demo/interactive testing of the system's coverage-related operations through the web API.

The system differentiates between a **coverage** and a **coverage configuration**. However, for most web API-related usages, you only need to work with coverages, as their related configuration is usually exposed together with each coverage.

Note the base URL starts with `http://webapp:5001`, as this jupyter notebook is expected to run inside the development docker compose stack. In a regular situation, the base URL would instead be something like `https://cline.arpa.veneto.it`

In [ ]:
%matplotlib ipympl

import logging

import httpx
import matplotlib.pyplot as plt
import pandas as pd

from arpav_ppcv.webapp.api_v2.schemas import (
    coverages,
    timeseries,
)

logging.basicConfig(level=logging.DEBUG)
logging.getLogger("httpcore").setLevel(logging.WARNING)
logging.getLogger("matplotlib").setLevel(logging.WARNING)

client = httpx.Client()
base_url = "http://webapp:5001/api/v2"

### Searching for coverages

The web API features a `coverages/coverages` endpoint, which can be used to list and search for coverages. This endpoint accepts a generic `possible_value` query parameter, which can be used to perform filtering.

In order to discover what values can be provided as `possible_value`, you can query the `coverages/configuration-parameters` API endpoint

In [ ]:
conf_params = client.get(f"{base_url}/coverages/configuration-parameters").json()["items"]
for cp in conf_params:
    print(f"{cp['name']}: {[pv['name'] for pv in cp['allowed_values']]}")

With knowlege about existing possible values, you can query the web API and retrieve relevant coverages. Let's examine coverages which have forecast TAS data. 

In this example we are looking for coverages that:

- are part of the `forecast` archive
- specify `tas` as their respective climatological variable
- have values aggregated by year
- represent `absolute` values

The API responds with a paginated list that includes 20 coverages and mentions that there are a total of 90 coverages in the following pages

In [ ]:
covs_response = client.get(
    f"{base_url}/coverages/coverages",
    params={
        "possible_value": [
            "archive:forecast",
            "climatological_variable:tas",
            "aggregation_period:annual",
            "measure:absolute",
        ]
    }
)
covs_response.raise_for_status()
parsed = covs_response.json()
meta = parsed["meta"]
print(meta)
items = parsed["items"]

We can inspect each item in the response, but just its `identifier` property is already enough to distinguish them. Also, note that each item in the response includes a URL for a corresponding detail response where we can get all the details

In [ ]:
for item in items:
    print(item["identifier"], item["url"], sep="\n", end="\n------\n")

Let's inspect the first item:

In [ ]:
for k, v in items[0].items():
    print(k)
    print(repr(v))
    print("-----")

As can be seen from inspecting this first item, the representation provided in the API response already includes some useful information like:

- WMS base URL and name of the main WMS layer
- Display name in both italian and english

Note that as an alternative to working with the simple python `dict`, we can also parse each item into a proper type by using the `arpav_ppcv` library which is part of the system. We will be using this alternate form, as it is easier to work with a typed data structure than with a generic dict.

In [ ]:
first_cov_short = coverages.LegacyForecastCoverageReadListItem(**items[0])

In [ ]:
first_cov_short.url, first_cov_short.identifier

### Retrieving details about a coverage

In order to get back other useful details we must now follow the items `url` property. For ease of use we will immediately convert from `dict` into the respective type, but remember this is not mandatory for working witht the API response:

In [ ]:
first_cov_detail_response = client.get(
    str(first_cov_short.url)
)
first_cov_detail_response.raise_for_status()

parsed = first_cov_detail_response.json()

first_cov = coverages.LegacyForecastCoverageReadDetail(**parsed)

Let's inspect the full details of the coverage:

In [ ]:
for k, v in first_cov.model_dump().items():
    print(k)
    print(repr(v))
    print("-------")

As can be seen, the coverage includes useful details like:

- data precision
- legend
- WMS-related URL and layer names
- data download URL
- etc.

The coverage's `identifier` is also useful for feeding into the `time-series` endpoint, which allows us to get back data for individual point locations

### Getting time series for a coverage

Let's take a reference location, and ask the web API for all the relevant time series.

The API allows specifying a number of query parameters:

- `coords` - a location specified as a WKT Point
- `datetime` - a temporal interval
- `include_coverage_data` - whether to include forecast time series
- `include_observation_data` - whether to also include a time series from the nearest observation station. Nearest in this context shall be the observation station which is nearest to the provided location, as long as it is not farther than the maximum distance
- `coverage_data_smoothing` and `observation_data_smoothing` - a set of post processing operations that can be applied to the raw time series data and which result in the generation of additional (derived) time series
- `include_coverage_uncertainty` and `include_coverage_related_data` - whether to include additional series, which are related to the coverage

In this case the response includes a total of 24 time series

In [ ]:
location = "POINT(12.3003 45.5826)"

time_series_response = client.get(
    f"{base_url}/coverages/forecast-time-series/{first_cov.identifier}",
    params={
        "coords": location,
        "datetime": "../..",
        "include_coverage_data": True,
        "include_observation_data": True,
        "coverage_data_smoothing": [
            "NO_SMOOTHING",
            "LOESS_SMOOTHING",
            "MOVING_AVERAGE_11_YEARS",
        ],
        "observation_data_smoothing": [
            "NO_SMOOTHING",
            "MOVING_AVERAGE_5_YEARS",
        ],
        "include_coverage_uncertainty": True,
        "include_coverage_related_data": True
    }
)
time_series_response.raise_for_status()
parsed = time_series_response.json()

In [ ]:
cov_series = []

for dict_series in parsed["series"]:
    time_series = timeseries.LegacyTimeSeries(**dict_series)
    print(time_series.name)
    cov_series.append(time_series)

Each time series has a number of useful properties in its `info`:

In [ ]:
for k, v in cov_series[0].info.items():
    print(k)
    print(repr(v))
    print("------")

The time series also has a `values` property, which is where the actual data is stored. It can be plotted easily by turning it into a pandas series:

In [ ]:
data_series = pd.Series(
    data=[v.value for v in cov_series[0].values], 
    index=[v.datetime for v in cov_series[0].values],
    name=cov_series[0].name
)

### Plotting time series

For the sake of keeping things tidy, let's create a function to obtain a pandas series, to be reused easily:

In [ ]:
def to_pandas_series(series: timeseries.LegacyTimeSeries) -> pd.Series:
    return pd.Series(
        data=[v.value for v in series.values],
        index=[v.datetime for v in series.values],
        name=series.name
    )

And now let's plot it

In [ ]:
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(ncols=2, nrows=2)
to_pandas_series(cov_series[0]).plot(ax=ax1, color="red")
to_pandas_series(cov_series[1]).plot(ax=ax2, color="green")
to_pandas_series(cov_series[2]).plot(ax=ax3, color="purple")
to_pandas_series(cov_series[3]).plot(ax=ax4)
plt.figlegend()

Let's thus plot all of the series which were returned:

In [ ]:
fig2, ax2 = plt.subplots()
for s in cov_series:
    pd_series = to_pandas_series(s)
    pd_series.name = "-".join(pd_series.name.split("-")[7:])
    pd_series.plot(ax=ax2)

fig2.legend(loc=7)
fig2.tight_layout()

Alternatively, let's plot only the model ensemble series

In [ ]:
fig3, ax3 = plt.subplots()
for s in cov_series:
    if "ensemble" in s.name:
        pd_series = to_pandas_series(s)
        pd_series.name = "-".join(pd_series.name.split("-")[8:])
        pd_series.plot(ax=ax3)
fig3.legend()